This notebook will demonstrate the following workflow:

* configure env
* (client) import and tag .pdf in VDI client, `tests/test_modeling/data/credit-protections-agreement.pdf`
* (client) export notes as labeled_data to .json, `tests/data/VDI_NotesData_v0.2.1.json`
* load .pdf into `Document`
* prepare `TextClassifier`
  - create training data
  - refine foundation models
* run models against `Document` sentences
* compare IOB results with labeled_data, `score_model_results`
* explore evaluation

## Configure Environment

In [2]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
import sys
sys.path.append('/workspaces/spa-vdi-3/pipelines')

import os
os.chdir('/workspaces/spa-vdi-3/pipelines/')


from src.modules.model_ensemble.utils import (
    load_txt,
    prepare_labels,
    score_model_results
)
from src.io.utils import xform_VDI_NotesData_to_page_labels
from src.modules.model_ensemble.Document import (
    Document,
    DocumentFactory
)




from src.modules.model_ensemble.TextClassifier import TextClassifier

from sentence_transformers import SentenceTransformer


ModuleNotFoundError: No module named 'src.modules.model_ensemble.Coordinator'

In [29]:

from src.modules.model_ensemble.Model import (
    Model,
    BinaryClassKeyWordModel,
    ClassificationModel
)

'''
from src.modules.model_ensemble.Coordinator import (
    SimplePassThruCoord,
    FirstHitCoord
)
from src.modules.model_ensemble.TextClassifier import TextClassifier

from sentence_transformers import SentenceTransformer
'''

ModuleNotFoundError: No module named 'src.modules.model_ensemble.Model'

In [ ]:
from pathlib import Path
import copy
import json

## Load .pdf

In [ ]:
filename = Path() / 'tests/test_modeling/data/credit-protection-agreement.pdf'
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
doc = DocumentFactory(filename, model, 1)
doc.get_sentences(page=0).__len__()

## Prepare `TextClassifier`

### Create training data

Some analysis must be done by comparing the targeted text with the surrounding text.  Reasonable differentiators should be obserable by humans.  This may be in teh form of key words, phrases, sentence structure, etc..

When initiating a project there may be limited training data.  For this example, the training data is manually created from experience, instead of directly obtaining data from source documents.

### Refine foundation models

In [ ]:
#config
config = {
    'TRAINING_DATA_DIR': {
        'model_topic': {
            'template1': Path('./models_data/template1'),
            'template2': Path('./models_data/template2'),
            'complaints': Path('./models_data/complaints'),
            'display': '...'
        }
    }
}

In [ ]:
#models and coordinator
model_name1 = 'template1'
model1 = BinaryClassKeyWordModel(
    model_name1,
    config['TRAINING_DATA_DIR']['model_topic'][model_name1]
)
model_name2 = 'complaints'
model2 = ClassificationModel(
    model_name2,
    config['TRAINING_DATA_DIR']['model_topic'][model_name2]
)
coord = FirstHitCoord()

#ensemble
tc = TextClassifier(
    name='model_topic',
    config=config,
    models=[model1,model2],
    coordinator=coord
)
checks = tc.validate_models_input()
checks

In [ ]:
results = tc.finetune_models()
results

## Run Models Against `Document` Sentences

In [ ]:
from src.io.utils import xform_VDI_NotesData_to_page_labels

vdi_notesdata_file = 'VDI_NotesData_v0.2.1.json'
filepath = Path(f'tests/data/{vdi_notesdata_file}')
with open(filepath, 'r') as f:
    notesdata = json.load(f)
labeled_data = xform_VDI_NotesData_to_page_labels(notesdata)

In [ ]:
#run inference
txt = doc.get_sentences(page=0)
pred_results = tc.run(txt)
pred_results

In [ ]:
#review scores
results = score_model_results(labeled_data, pred_results, doc)
results

## Compare IOB Results with `labeled_data`, `score_model_results`
## Explore Evaluation